#Criação da tabela bronze

## Objetivos

- Leitura de dados da API
- Criação de dataframe em SPARK
- Conversão do dataframe em tabela

In [0]:

import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql.functions import *

In [0]:
api_key = dbutils.fs.head('dbfs:/Volumes/api_football/default/secret/api_key.txt')
#print(api_key)

In [0]:
url = (
    f'https://apiv3.apifootball.com/'
    f'?action=get_events'
    f'&from=2026-01-01'
    f'&to=2026-12-31'
    f'&league_id=99' #brasil
    f'&APIkey={api_key}'
)
#request
response = requests.get(url)

#print(response.status_code)

In [0]:
#repassa os dados para um arquivo json
data = response.json()
#criação do dataframe em pandas
df_raw = pd.DataFrame(data)
#conversão em Spark DataFrame
df_raw = spark.createDataFrame(df_raw)

#criação de coluna para indicar o horário de processamento dos dados no fuso de SP (GMT -3)
dh_processing_br = datetime.now() - timedelta(hours=3)
df_raw = df_raw.withColumn(
    'dh_processing_br', lit(dh_processing_br)
    )
    
#df_raw.createOrReplaceTempView('vw_raw')

In [0]:
df_clean = df_raw.withColumn("lineup", col("lineup") #coluna lineup
    .withField("away", col("lineup.away").dropFields("missing_players")) #campo away
    .withField("home", col("lineup.home").dropFields("missing_players")) #campo home
    ) 

In [0]:
df_clean.write.mode("append").saveAsTable('api_football.bronze.matches_raw')

In [0]:
%sql
--select * from api_football.bronze.matches_raw